# 🟢 미션 1 (필수) — 내 문장으로 어텐션을 계산하고 파이토치와 맞춰 본다

**제출**
1. `일치: True` 가 나온 화면 캡처
2. 한 줄: 내 문장에서 **Q·K·V 가 각각 무엇이었는지**

In [1]:
import sys

# 노트북(cwd=이 폴더)에서도, .py 를 상위에서 돌려도 attn/koplot 을 찾게 한다.
# ⚠️ __file__ 은 주피터 커널에 정의되지 않는다 — 노트북에서 NameError 로 죽는다.
sys.path[:0] = ["..", "."]

import torch

import attn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## 1. 내 문장을 정한다

토큰 4~6개. 띄어쓰기 단위로 잘라 쓰면 된다.

In [2]:
# TODO: 내 문장으로 바꾸기
내토큰 = ["오늘", "날씨가", "정말", "좋다"]

L = len(내토큰)
d = 8                                  # 각 토큰을 나타내는 벡터의 길이
x = torch.randn(L, d)                  # 학습 안 된 임의의 벡터 (오늘은 이걸로 충분)

print("문장:", " ".join(내토큰))
print("x shape:", tuple(x.shape), " ← (토큰 수, 벡터 길이)")

문장: 오늘 날씨가 정말 좋다
x shape: (4, 8)  ← (토큰 수, 벡터 길이)


## 2. 셀프 어텐션을 통과시킨다

In [3]:
m = attn.SelfAttention(d)
내출력, 내격자 = m(x)

print("출력 shape:", tuple(내출력.shape), " ← 들어간 모양 그대로")
print()
attn.show_grid(내격자, 내토큰)

출력 shape: (4, 8)  ← 들어간 모양 그대로

         |     오늘   날씨가     정말     좋다   (행 합)
    오늘 |   0.3427   0.2065   0.2639   0.1869    1.0000
  날씨가 |   0.2487   0.2547   0.2135   0.2831    1.0000
    정말 |   0.2421   0.2337   0.2720   0.2522    1.0000
    좋다 |   0.2440   0.2743   0.2231   0.2586    1.0000


행마다 합이 1인지 확인한다.

⚠️ 격자의 **값이 무슨 뜻인지는 보지 않는다.** 학습을 하지 않았으므로 의미가 없다.

## 3. Q·K·V 가 각각 무엇인지 확인한다

In [4]:
Q, K, V = m.Wq(x), m.Wk(x), m.Wv(x)
for 이름, t in [("Q (질의)", Q), ("K (이름표)", K), ("V (내용)", V)]:
    print(f"{이름:<12s} {tuple(t.shape)}")

print()
print("Q 와 K 가 다른가        :", not torch.allclose(Q, K))
print("K 와 V 가 다른가        :", not torch.allclose(K, V))
print("셋 다 같은 x 에서 나왔나 : True   ← 이게 '셀프' 어텐션이다")

Q (질의)       (4, 8)
K (이름표)      (4, 8)
V (내용)       (4, 8)

Q 와 K 가 다른가        : True
K 와 V 가 다른가        : True
셋 다 같은 x 에서 나왔나 : True   ← 이게 '셀프' 어텐션이다


## 4. ★ 파이토치와 같은 값이 나오는가

같은 가중치를 `nn.MultiheadAttention` 에 심고 비교한다.

In [5]:
mha = attn.plant_into_torch(m, d)
토치출력, 토치격자 = attn.run_torch(mha, x)

일치 = torch.allclose(내출력, 토치출력, atol=1e-5)
print("내 계산  :", [round(v, 5) for v in 내출력[0].tolist()[:4]], "...")
print("파이토치 :", [round(v, 5) for v in 토치출력[0].tolist()[:4]], "...")
print()
print("일치:", 일치)
print("최대 오차:", f"{(내출력 - 토치출력).abs().max():.2e}")

내 계산  : [0.26216, 0.44653, 0.04624, -0.07912] ...
파이토치 : [0.26216, 0.44653, 0.04624, -0.07912] ...

일치: True
최대 오차: 2.98e-08


오차가 `1e-8` ~ `1e-7` 근처로 나오는 것이 정상이다. 0이 아니어도 맞은 것이다.
부동소수점 계산은 순서가 조금만 달라도 마지막 자리가 흔들린다.
그래서 `==` 가 아니라 `torch.allclose` 로 판정한다.

## 5. 제출할 한 줄을 적는다

In [6]:
# TODO: 아래 문장을 내 말로 채운다
print("""
내 문장에서
  Q 는 ___________________
  K 는 ___________________
  V 는 ___________________
""")


내 문장에서
  Q 는 ___________________
  K 는 ___________________
  V 는 ___________________

